# Fides — inference-only verification tap

Reference implementations built against the AI 2040: Plan A verification SITREP (amododesign.com/ai-verification/plan-a-sitrep) plus faithful, from-source reimplementations of TOPLOC and Token-DiFR.

This notebook exists for one specific reason: the environment Fides was originally built in has no GPU, so `pip install toploc` could be installed but its compiled C extension failed with a real ABI mismatch against the available torch build (documented in `spec/PROTOCOL.md` section 11). Colab and Kaggle ship maintained, tested torch+CUDA stacks, so this is worth trying here — if it works, you can compare Fides's own from-scratch reimplementations directly against the real published packages instead of just against each other.

**Before running:** push Fides to your own GitHub repo, then paste the URL into the first code cell below.

In [ ]:
# EDIT THIS: your GitHub repo URL after you've pushed Fides
REPO_URL = "https://github.com/AnonRish/fides.git"

!git clone $REPO_URL fides_repo
%cd fides_repo
!pip install -e ".[dev]" -q
print("Installed.")

## 1. Run the full test suite

Everything here is pure Python plus `blake3` and `cryptography` — no GPU needed for any of this part.

In [ ]:
!pytest tests/ -v

## 2. Run the end-to-end demo

Honest device, a blatant liar, a subtle covert-training attack, and the sampling-vs-ZK-accumulator comparison.

In [ ]:
!python examples/demo.py

## 3. Try the real TOPLOC package

This is the part that couldn't be tested in the original sandbox. If this cell's imports succeed here, run the real README example from github.com/PrimeIntellect-ai/toploc and compare its output against `fides.toploc_reference`'s reimplementation on the same synthetic data.

In [ ]:
!pip install toploc -q
try:
    import torch
    from toploc import build_proofs_base64, verify_proofs_base64
    print("Real toploc package imported successfully — this failed in the original sandbox (ABI mismatch).")
    TOPLOC_AVAILABLE = True
except Exception as e:
    print(f"Import failed: {e}")
    print("If you're on a CPU-only Colab runtime, switch to a GPU runtime (Runtime > Change runtime type) and re-run this cell.")
    TOPLOC_AVAILABLE = False

In [ ]:
if TOPLOC_AVAILABLE:
    torch.manual_seed(42)
    prefill = [torch.randn(5, 16, dtype=torch.bfloat16)]
    generate = [torch.randn(16, dtype=torch.bfloat16) for _ in range(10)]
    activations = prefill + generate

    proofs = build_proofs_base64(activations, decode_batching_size=3, topk=4, skip_prefill=False)
    print("Real TOPLOC proofs:", proofs)

    results = verify_proofs_base64(activations, proofs, decode_batching_size=3, topk=4, skip_prefill=False)
    print("\nVerification against identical activations (should all match):")
    for r in results:
        print(" ", r)

    jittered = [a * 1.01 for a in activations]
    results_j = verify_proofs_base64(jittered, proofs, decode_batching_size=3, topk=4, skip_prefill=False)
    print("\nVerification against 1% jittered activations:")
    for r in results_j:
        print(" ", r)

## 4. Fides's own reimplementation, same idea, synthetic data

`fides.toploc_reference` was built by reading TOPLOC's `poly.py` directly (top-k-by-magnitude selection + polynomial interpolation over GF(65537)), not by running the real package — this environment is the first place that's actually been possible. Compare the *shape* of the results, not exact values (different modulus/packing, by design — see the module docstring for why).

In [ ]:
from fides.toploc_reference import build_proof, verify_proof
import random

rng = random.Random(1)
values = [rng.gauss(0, 1) for _ in range(200)]
proof = build_proof(values, k=16)

jrng = random.Random(2)
jittered = [v + jrng.gauss(0, 0.0005) for v in values]
honest_result = verify_proof(proof, jittered)
print("Fides reimplementation, honest jitter:", honest_result)

frng = random.Random(3)
fake = [frng.gauss(0, 1) for _ in range(200)]
fake_result = verify_proof(proof, fake)
print("Fides reimplementation, fabricated:   ", fake_result)

## 5. Token-DiFR reimplementation (no GPU needed at all for this one)

Built from reading `github.com/adamkarvonen/difr`'s `token_difr_vllm.py` directly — seed-synchronized Gumbel-max sampling. This part needs no torch and works identically here or in the original sandbox; included for completeness.

In [ ]:
from fides.difr_reference import (
    synthetic_distribution, quantize_distribution, gumbel_max_sample, verify_sequence, match_rate
)

vocab_size, seed = 300, 5
distributions = [synthetic_distribution(vocab_size, seed=seed + p) for p in range(200)]
claimed = [gumbel_max_sample(d, seed=seed, position=p) for p, d in enumerate(distributions)]

for noise_scale in (0.0, 0.5, 1.5, 4.0):
    perturbed = [quantize_distribution(d, noise_scale=noise_scale, seed=500 + p) for p, d in enumerate(distributions)]
    results = verify_sequence(claimed, perturbed, seed=seed)
    print(f"noise_scale={noise_scale:4.1f}  match_rate={match_rate(results):.3f}")

## Notes

- If section 3 worked here, that's a genuinely new result relative to the repo's own documentation — worth updating `spec/PROTOCOL.md` section 11 with what actually happened (success or a different failure mode), rather than leaving the sandbox's ABI-mismatch finding as the last word.
- Sections 4 and 5 don't need a GPU and will behave identically to running them locally — they're here for convenience and side-by-side comparison, not because Colab adds anything for them specifically.
- Kaggle notebooks work the same way: create a new notebook, enable internet access in the settings panel, and paste these same cells in.